# Aprendizaje Federado con Flower + Docker + Seguridad (trabajo adicional)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA · Sesión 5**

Versión con **Flower (`flwr`)** del tutorial federado con Docker. Es el **framework de producción** para aprendizaje federado: organiza el mismo FedAvg que vimos, en un `NumPyClient` (entrena local) y una estrategia `FedAvg` en el servidor. Le mantenemos el **plus de seguridad**: un **ataque de envenenamiento de datos** (label flipping 3 ↔ 7) para ver su efecto.

Convive con los otros dos notebooks de la sesión:
- **`federated_tutorial.ipynb`** — FedAvg desde cero sin Docker, para entender el algoritmo.
- **`federated_envenenamiento.ipynb`** —  Mismo algoritmo + envenenamiento.
- **este** — igual, pero con **Flower** (el framework real) dentro de Docker.

## Por qué Flower (y no TFF)

**TensorFlow Federated** ya no instala desde PyPI (fija `jaxlib`/`farmhashpy` que fueron retirados), así que su `docker build` falla. **Flower** sí instala limpio y es hoy el estándar más usado para federado: es **agnóstico del modelo** (numpy, PyTorch, TensorFlow, scikit-learn) y sirve tanto para **simular** (lo de hoy) como para **desplegar** clientes reales.

Aquí reutilizamos **el mismo modelo softmax en numpy** como entrenamiento local del cliente, y dejamos que **Flower orqueste** la federación: reparte el modelo, recoge los pesos y los agrega con `FedAvg`. La simulación de Flower usa **Ray** por debajo (por eso la imagen es algo más grande y el primer build tarda un poco más).

## Contexto: seguridad en aprendizaje federado

El servidor **nunca ve los datos**, solo **parámetros**. Pero un cliente **malicioso** controla su entrenamiento local y puede enviar actualizaciones **envenenadas**. Hacemos un **label flipping dirigido**: el cliente 0 intercambia **3 ↔ 7**. Al agregar con FedAvg, el modelo global “aprende” esa confusión: **caen las clases 3 y 7**, mientras la precisión global baja de forma más sutil. Por eso hay que mirar la métrica **por clase**. Conecta con la **Sesión 7**.

## Paso 1 — Estructura del proyecto

```
.
├── tutorial_federado_flower.py   # cliente + servidor Flower (FedAvg), limpio vs envenenado
├── pyproject.toml                # dependencias (uv / PEP 621): flwr[simulation] + sklearn
└── Dockerfile                    # imagen reproducible
```

## Paso 2 — `tutorial_federado_flower.py`

Piezas de Flower:
- **`FLClient(NumPyClient)`** — `fit()` recibe los pesos globales, hace SGD local y devuelve los pesos (nunca los datos).
- **`ClientApp`** — crea el cliente según su `partition-id`. Usamos **dos** ClientApps (datos limpios y envenenados) para evitar el problema de aislamiento de procesos de Ray: cada uno referencia su partición fija.
- **`ServerApp` + estrategia `FedAvg`** — agrega los pesos y, con `evaluate_fn`, evalúa el modelo global **en el servidor** cada ronda (precisión global y por clase).
- **`run_simulation(...)`** — corre todo en un proceso, con `NUM_CLIENTS` nodos.

Corre en segundos por corrida; entrena una vez sin ataque y otra con el cliente 0 envenenado, y guarda el gráfico.

In [ ]:
%%writefile tutorial_federado_flower.py
"""
Aprendizaje federado con Flower (flwr) + ataque de envenenamiento de datos (MLOps II).

Mismo experimento que la version numpy, pero orquestado con Flower: cada cliente
es un NumPyClient (entrena local) y el servidor agrega con la estrategia FedAvg.
Se entrena SIN ataque y CON un cliente que intercambia etiquetas 3<->7.
Pensado para ejecutarse dentro de Docker; guarda un grafico comparativo.
"""
import copy
import numpy as np
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use("Agg")  # backend sin pantalla (headless, para el contenedor)
import matplotlib.pyplot as plt

import flwr as fl
from flwr.common import Context, ndarrays_to_parameters
from flwr.client import NumPyClient, ClientApp
from flwr.server import ServerApp, ServerAppComponents, ServerConfig
from flwr.server.strategy import FedAvg

# ---- Hiperparametros ----
NUM_CLIENTS = 3
ROUNDS = 20
LOCAL_EPOCHS = 15
LR = 0.5
SEED = 0
rng = np.random.RandomState(SEED)

# ---- Datos y modelo softmax (mismo que la version numpy) ----
X, y = load_digits(return_X_y=True)
X = StandardScaler().fit_transform(X)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
N_FEAT, N_CLS = X.shape[1], 10

def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True); e = np.exp(Z); return e / e.sum(axis=1, keepdims=True)
def onehot(y):
    O = np.zeros((len(y), N_CLS)); O[np.arange(len(y)), y] = 1; return O
def paso_sgd(W, b, X, y, lr):
    P = softmax(X @ W + b); D = (P - onehot(y)) / len(y)
    return W - lr * (X.T @ D), b - lr * D.sum(axis=0)
def evaluar(W, b):
    pred = np.argmax(Xte @ W + b, axis=1)
    acc = (pred == yte).mean()
    pc = {c: (pred[yte == c] == c).mean() if (yte == c).sum() else 0.0 for c in range(N_CLS)}
    return acc, pc

# ---- Particiones: se calculan al importar el modulo (visibles para los workers de Ray) ----
def _particionar(K):
    idx = rng.permutation(len(Xtr))
    return [[Xtr[c].copy(), ytr[c].copy()] for c in np.array_split(idx, K)]
def _envenenar(cli):
    Xc, yc = cli; yc = yc.copy(); m3, m7 = (yc == 3), (yc == 7); yc[m3], yc[m7] = 7, 3
    return [Xc, yc]

CLEAN_PARTS = _particionar(NUM_CLIENTS)
POISON_PARTS = copy.deepcopy(CLEAN_PARTS); POISON_PARTS[0] = _envenenar(POISON_PARTS[0])

# ---- Cliente Flower: entrena local y devuelve pesos (nunca datos) ----
class FLClient(NumPyClient):
    def __init__(self, data):
        self.Xc, self.yc = data
    def fit(self, parameters, config):
        W, b = parameters
        for _ in range(LOCAL_EPOCHS):
            W, b = paso_sgd(W, b, self.Xc, self.yc, LR)
        return [W, b], len(self.Xc), {}
    def evaluate(self, parameters, config):
        return 0.0, len(self.Xc), {}  # la evaluacion la hace el servidor

def _client_fn(parts):
    def client_fn(context: Context):
        cid = int(context.node_config["partition-id"])
        return FLClient(parts[cid]).to_client()
    return client_fn

# ---- Servidor: FedAvg + evaluacion centralizada por ronda ----
def _make_eval(hist, last_pc):
    def evaluate_fn(server_round, parameters, config):
        W, b = parameters
        acc, pc = evaluar(W, b)
        hist.append(acc); last_pc.clear(); last_pc.update(pc)
        return 1.0 - acc, {"accuracy": acc}
    return evaluate_fn

def _server_fn(hist, last_pc):
    def server_fn(context: Context):
        init = ndarrays_to_parameters([np.zeros((N_FEAT, N_CLS)), np.zeros(N_CLS)])
        strat = FedAvg(fraction_fit=1.0, fraction_evaluate=0.0,
                       min_fit_clients=NUM_CLIENTS, min_available_clients=NUM_CLIENTS,
                       initial_parameters=init, evaluate_fn=_make_eval(hist, last_pc))
        return ServerAppComponents(strategy=strat, config=ServerConfig(num_rounds=ROUNDS))
    return server_fn

def correr(parts, etiqueta):
    hist, last_pc = [], {}
    server_app = ServerApp(server_fn=_server_fn(hist, last_pc))
    client_app = ClientApp(client_fn=_client_fn(parts))
    print(f"\n=== Flower: {etiqueta} ===")
    fl.simulation.run_simulation(
        server_app=server_app, client_app=client_app,
        num_supernodes=NUM_CLIENTS,
        backend_config={"client_resources": {"num_cpus": 1}},
    )
    return hist, last_pc

if __name__ == "__main__":
    hist_clean, _ = correr(CLEAN_PARTS, "SIN ataque")
    hist_poison, pc_poison = correr(POISON_PARTS, "CON ataque (cliente 0: 3<->7)")
    acc_clean = hist_clean[-1] if hist_clean else float("nan")
    acc_poison = hist_poison[-1] if hist_poison else float("nan")
    print(f"\nPrecision global -> sin ataque: {acc_clean:.3f} | con ataque: {acc_poison:.3f}")
    print("\nPrecision por clase (con ataque):")
    for c in range(N_CLS):
        marca = "   <- objetivo del ataque" if c in (3, 7) else ""
        print(f"  Clase {c}: {pc_poison.get(c, 0):.2f}{marca}")
    n = min(len(hist_clean), len(hist_poison))
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, n + 1), hist_clean[:n], label=f"Sin ataque ({acc_clean:.2f})")
    plt.plot(range(1, n + 1), hist_poison[:n], "--", label=f"Con ataque ({acc_poison:.2f})")
    plt.xlabel("Ronda de comunicacion"); plt.ylabel("Precision global en test")
    plt.title("Flower + FedAvg: efecto del envenenamiento de datos")
    plt.legend(); plt.grid(True); plt.tight_layout()
    plt.savefig("envenenamiento_precision.png", dpi=120)
    print("\nGrafico guardado en envenenamiento_precision.png")


## Paso 3 — `pyproject.toml` (uv)

> **Nota para quienes usan Poetry.** El proyecto usa **uv**. Con Poetry (≥ 2.0, PEP 621) es equivalente: `poetry install` y `poetry run python tutorial_federado_flower.py`; en el `Dockerfile`, reemplaza `uv pip install --system ...` por `RUN pip install poetry && poetry config virtualenvs.create false && poetry install --no-root`.

In [ ]:
%%writefile pyproject.toml
[project]
name = "mlops-ii-federado-flower"
version = "0.1.0"
description = "Aprendizaje federado con Flower + demo de envenenamiento (MLOps II)"
requires-python = ">=3.9,<3.12"
dependencies = [
    "flwr[simulation]>=1.13,<2",
    "numpy>=1.24",
    "scikit-learn>=1.3",
    "matplotlib>=3.7",
]

[tool.uv]
package = false


## Paso 4 — `Dockerfile` (uv)

Se instala en el **Python del sistema** (`uv pip install --system`), no en un `.venv` dentro de `/app`, para que el montaje `-v ${pwd}:/app` del `run` no oculte las dependencias. La imagen incluye **Flower + Ray**, así, el primer `build` tarda unos minutos.

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

# uv: gestor de paquetes rapido (Astral).
COPY --from=ghcr.io/astral-sh/uv:latest /uv /uvx /bin/

WORKDIR /app
COPY pyproject.toml .

# Flower con motor de simulacion (usa Ray). Se instala en el Python del sistema
# para que el montaje -v ${pwd}:/app del run no oculte las dependencias.
RUN uv pip install --system "flwr[simulation]>=1.13,<2" numpy scikit-learn matplotlib

COPY tutorial_federado_flower.py .
CMD ["python", "tutorial_federado_flower.py"]


## Paso 5 — Construir y ejecutar

Con **Docker Desktop** abierto, desde la carpeta del proyecto:

```bash
docker build -t federated-flower .
```

Luego, montando la carpeta local (para recuperar el gráfico). En **una sola línea** según tu terminal:

**PowerShell:**
```powershell
docker run --rm -v ${pwd}:/app federated-flower
```

**CMD:**
```cmd
docker run --rm -v "%cd%":/app federated-flower
```

Verás logs de Flower/Ray (algún `WARNING`/`Deprecated` es normal) y, al final, la precisión por clase. El gráfico queda en **`envenenamiento_precision.png`** en tu carpeta.

## Paso 6 — Qué observar en los resultados

Corrida de referencia (3 clientes, uno envenenado):

- **Global:** ~0.97 sin ataque → ~0.95 con ataque (caída sutil).
- **Por clase:** **3 y 7 se desploman** (≈1.00 → ≈0.89), el resto se mantiene. Firma del ataque dirigido.

**Lección:** monitorear solo la métrica global no alcanza; el **monitoreo por clase/segmento** delata el envenenamiento.

## Paso 7 — Defensas (el plus de seguridad)

Se profundiza en la **Sesión 7**. En Flower, varias se implementan cambiando la **estrategia** del servidor:

- **Agregación robusta:** en vez de `FedAvg`, usar estrategias resistentes (mediana / trimmed mean / **Krum**) que descartan updates atípicas.
- **Detección de anomalías:** medir la distancia de cada update al consenso y filtrar las que se desvían.
- **Clipping + privacidad diferencial:** limitar/ruidear la contribución de cada cliente (Flower tiene wrappers de DP).
- **Validación / reputación de clientes:** evaluar cada update contra un set limpio del servidor.
- **Monitoreo por clase con alertas:** detectar caídas dirigidas apenas ocurren.

**Idea final:** federar mejora la privacidad, no la seguridad por sí solo. Se necesita **agregación robusta + monitoreo + gobernanza**.

---

### Cómo encaja
- **Sin Docker** (`federated_tutorial.ipynb`): entender FedAvg desde cero.
- **Sin Docker (numpy)** (`federated_envenenamiento.ipynb`): reproducibilidad + seguridad, mínimo de dependencias.
- **Con Docker (Flower)** (este): el **framework real** de federado + seguridad.
- **Mini-TP 5** e **integrador**: si tu proyecto usa federado, mostrar Flower + una defensa es un gran diferencial.